# Biomarker Analysis Pipeline (v4)

Orchestrates the end-to-end biomarker pipeline. Each stage is dispatched as a subprocess that uses the active kernel's Python (`sys.executable`), so all stages share the conda environment of this notebook *without* leaking variables between them. Set the `RUN_*` toggles below to skip stages you don't need to re-execute.

### Cohorts
- **Cohort 1** (`cohort1`) — First-line ICI vs all never-ICI, unmatched
- **Cohort 2** (`cohort2`) — Lines 1–3, 1:1 matched on (cancer_type, line_category)

### Propensity-score models (trained within each cohort)
- **covariates_only** — elastic net CV LR on demographics + cancer type + line
- **covariates_plus_embeddings** — elastic net CV LR on covariates + text embeddings

### Analysis tracks (using cohort-specific PS without adjustment)
- **Track 1** — ICI-only, prognostic: `S(t) ~ base_vars + line_dummies + marker`
  - **unweighted** + **ATE** (1/ps generalizability weights)
- **Track 2** — Full cohort, predictive interaction: `S(t) ~ base_vars + line_dummies + marker + ICI + marker x ICI`
  - **noIPTW** + **ATE** weights

### Stages
1. **Cohort construction** — `build_line_matched_cohort.py` (produces both cohorts)
2. **Embedding generation** — `ICI_generate_embeddings.py` (pools note embeddings per cohort)
3. **Propensity scores** — `ICI_train_propensity.py` (trains both PS models per cohort)
4. **IPTW datasets** — `generate_IPTW_df.py` (loops over all cohort × ps_model combinations)
5. **Cox models** — `run_IPTW_analysis.py` (Track 1 + Track 2 per cohort × ps_model)
6. **Compile results** — `compile_IPTW_results.py` (aggregates significant hits)

All cohort / PS-model looping is handled inside each script.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "python_scripts").is_dir() and (candidate / "jupyter_notebooks").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
print(f"Repo root: {REPO_ROOT}")
print(f"Python:    {sys.executable}")

# Toggle individual stages off when re-running a partial pipeline (e.g. after
# only edits to compile_IPTW_results.py you can set every flag except
# RUN_COMPILE to False).
RUN_COHORTS = True   # stage 1: build_line_matched_cohort.py
RUN_EMBED   = True   # stage 2: ICI_generate_embeddings.py
RUN_PS      = True   # stage 3: ICI_train_propensity.py
RUN_IPTW    = True   # stage 4: generate_IPTW_df.py
RUN_COX     = True   # stage 5: run_IPTW_analysis.py
RUN_COMPILE = True   # stage 6: compile_IPTW_results.py

In [ ]:
BIO_DIR = "python_scripts/biomarker_analysis"

STAGES: list[tuple[str, bool, list[str]]] = [
    ("Stage 1: Cohort construction",   RUN_COHORTS, [f"{BIO_DIR}/build_line_matched_cohort.py"]),
    ("Stage 2: Embedding generation",  RUN_EMBED,   [f"{BIO_DIR}/ICI_generate_embeddings.py"]),
    ("Stage 3: Propensity scores",     RUN_PS,      [f"{BIO_DIR}/ICI_train_propensity.py"]),
    ("Stage 4: IPTW datasets",         RUN_IPTW,    [f"{BIO_DIR}/generate_IPTW_df.py"]),
    ("Stage 5: Cox model analysis",    RUN_COX,     [f"{BIO_DIR}/run_IPTW_analysis.py"]),
    ("Stage 6: Compile results",       RUN_COMPILE, [f"{BIO_DIR}/compile_IPTW_results.py"]),
]


def run_stage(label: str, args: list[str]) -> None:
    print(f"\n=== {label}: {' '.join(args)} ===", flush=True)
    subprocess.run([sys.executable, *args], cwd=REPO_ROOT, check=True)


for label, enabled, args in STAGES:
    if not enabled:
        print(f"\n=== SKIPPED  {label} ===", flush=True)
        continue
    run_stage(label, args)

print("\nDone. Compiled hits land in biomarker_analysis/compiled_results/.", flush=True)
print("Re-run prep_figure_5.py + plot_figure_5_biomarkers.py to refresh Figure 5.", flush=True)

## Per-stage notes

Each script is notebook-safe (no `argparse`, no `sys.exit`) and re-runs idempotently. If you need to inspect intermediate outputs while the pipeline is running, the data layout is:

| Stage | Writes to |
|-------|-----------|
| 1 | `biomarker_analysis/matched_cohorts/matched_cohort_{cohort1,cohort2}.csv.gz` |
| 2 | `biomarker_analysis/embeddings/{cohort}/w_{buffer}_day_buffer/` |
| 3 | `treatment_prediction/{cohort}/{ps_model}_propensity/w_{buffer}_day_buffer/predictions.csv.gz` |
| 4 | `biomarker_analysis/IPTW_df_{cohort}_{ps_model}.csv.gz` |
| 5 | `biomarker_analysis/IPTW_runs_{cohort}_{ps_model}/*_track{1,2}_{ATE,noIPTW,unweighted}_*.csv.gz` |
| 6 | `biomarker_analysis/compiled_results/track{1,2}_all_significant_hits.csv.gz` |

The `prep_figure_5.py` script in `jupyter_notebooks/manuscript_figures/data_generation/` consumes stage-3 predictions (panel A) and stage-6 compiled hits (panels B/C).